# OceanTrace AI — SegFormer Vision Transformer Training
## Marine Oil Spill & Look-Alike Segmentation (SAR & Optical Satellite Imagery)
**Smart India Hackathon 2026 | Problem Statement SIH26143**

This notebook contains the complete, reproducible deep learning training pipeline for semantic segmentation of marine oil spills.
- **Model Architecture**: SegFormer-B2 (Mix Transformer `nvidia/mit-b2` encoder + All-MLP Decoder)
- **Target Classes**: 0: Clean Sea, 1: Oil Spill, 2: Look-Alike, 3: Ship, 4: Land
- **Primary Dataset**: M4D Marine Oil Spill Detection Dataset (SAR)
- **Secondary Dataset**: GlobalOSD-SAR
- **Hardware Target**: Kaggle GPU (T4 / P100) or Google Colab GPU (T4 / A100)

In [ ]:
# Section 0: Dependency Installation
!pip install -q torch torchvision transformers==4.40.0 accelerate datasets opencv-python-headless scikit-learn matplotlib seaborn tqdm pyyaml

In [ ]:
# Section 1: GPU Verification & Environment Report
import os
import sys
import torch

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Active GPU: {gpu_name}")
    print(f"GPU Memory: {total_mem:.2f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    device = torch.device("cpu")
    print("WARNING: No GPU detected! Running on CPU. For training, enable GPU runtime.")

In [ ]:
# Section 2: Hyperparameter & Training Configuration
from dataclasses import dataclass, field
from typing import List

@dataclass
class TrainingConfig:
    # Paths (adjust depending on platform: /kaggle/input/... or /content/drive/...)
    data_dir: str = "/kaggle/input/m4d-oil-spill"
    output_dir: str = "./artifacts/checkpoints"
    metrics_dir: str = "./artifacts/metrics"
    
    # Model Architecture
    pretrained_model_name: str = "nvidia/mit-b2"
    num_classes: int = 5
    image_size: int = 512
    
    # Class Metadata
    class_names: List[str] = field(default_factory=lambda: ["Clean Sea", "Oil Spill", "Look-Alike", "Ship", "Land"])
    # Effective weighting to handle extreme class imbalance (oil spill pixels are sparse)
    class_weights: List[float] = field(default_factory=lambda: [0.35, 10.0, 4.0, 3.0, 1.5])
    
    # Optimization & Training Loop
    batch_size: int = 4
    epochs: int = 40
    learning_rate: float = 6e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.08
    use_amp: bool = True
    early_stopping_patience: int = 8
    random_seed: int = 42

config = TrainingConfig()
os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(config.metrics_dir, exist_ok=True)
print("Training configuration successfully initialized:", config)

In [ ]:
# Section 3: Dataset Class & Preprocessing Pipeline
import cv2
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class OilSpillDataset(Dataset):
    """
    PyTorch Dataset for Sentinel-1 SAR & Optical Satellite Oil Spill Segmentation.
    Loads SAR images (converting 1-channel to 3-channel for Vision Transformer compatibility)
    and normalizes with ImageNet parameters.
    """
    def __init__(self, image_paths, mask_paths=None, image_size=512, augment=False):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.image_size = image_size
        self.augment = augment
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = Image.open(img_path).convert("RGB")
        img = img.resize((self.image_size, self.image_size), Image.BILINEAR)
        img_np = np.array(img, dtype=np.float32) / 255.0

        if self.mask_paths is not None:
            mask_path = self.mask_paths[idx]
            mask = Image.open(mask_path).convert("L")
            mask = mask.resize((self.image_size, self.image_size), Image.NEAREST)
            mask_np = np.array(mask, dtype=np.int64)
            # Map mask to target classes if values exceed num_classes
            mask_np = np.clip(mask_np, 0, config.num_classes - 1)
        else:
            mask_np = np.zeros((self.image_size, self.image_size), dtype=np.int64)

        if self.augment:
            if np.random.rand() > 0.5:
                img_np = np.fliplr(img_np).copy()
                mask_np = np.fliplr(mask_np).copy()
            if np.random.rand() > 0.5:
                img_np = np.flipud(img_np).copy()
                mask_np = np.flipud(mask_np).copy()
            k = np.random.choice([0, 1, 2, 3])
            if k > 0:
                img_np = np.rot90(img_np, k).copy()
                mask_np = np.rot90(mask_np, k).copy()

        # Normalize
        norm_img = (img_np - self.mean) / self.std
        tensor_img = torch.from_numpy(norm_img.transpose(2, 0, 1)).float()
        tensor_mask = torch.from_numpy(mask_np).long()

        return {"pixel_values": tensor_img, "labels": tensor_mask}

print("OilSpillDataset pipeline compiled.")

In [ ]:
# Section 4: Loss Functions with Class Imbalance Compensation
import torch.nn as nn
import torch.nn.functional as F

class CombinedDiceCELoss(nn.Module):
    """
    Combines Weighted Cross-Entropy Loss with Multi-Class Soft Dice Loss.
    Ensures accurate boundary delineation for thin oil sheens and heavy emulsion patches.
    """
    def __init__(self, class_weights=None, ce_weight=0.6, dice_weight=0.4, eps=1e-6):
        super().__init__()
        self.ce_weight = ce_weight
        self.dice_weight = dice_weight
        self.eps = eps
        if class_weights is not None:
            self.ce = nn.CrossEntropyLoss(weight=class_weights)
        else:
            self.ce = nn.CrossEntropyLoss()

    def forward(self, logits, targets):
        # Upsample logits to target mask dimensions
        if logits.shape[-2:] != targets.shape[-2:]:
            logits = F.interpolate(logits, size=targets.shape[-2:], mode="bilinear", align_corners=False)
        
        ce_loss = self.ce(logits, targets)
        
        # Soft Dice Loss calculation
        probs = F.softmax(logits, dim=1)
        num_classes = logits.shape[1]
        targets_one_hot = F.one_hot(targets, num_classes=num_classes).permute(0, 3, 1, 2).float()
        
        dims = (0, 2, 3)
        intersection = torch.sum(probs * targets_one_hot, dims)
        cardinality = torch.sum(probs + targets_one_hot, dims)
        dice_score = (2.0 * intersection + self.eps) / (cardinality + self.eps)
        dice_loss = 1.0 - torch.mean(dice_score)
        
        return self.ce_weight * ce_loss + self.dice_weight * dice_loss

print("CombinedDiceCELoss module compiled.")

In [ ]:
# Section 5: Model Initialization (SegFormer-B2 with MiT-B2 Vision Transformer Backbone)
from transformers import SegformerForSemanticSegmentation, SegformerConfig

id2label = {i: name for i, name in enumerate(config.class_names)}
label2id = {name: i for i, name in id2label.items()}

print(f"Initializing SegFormer model from {config.pretrained_model_name}...")
model = SegformerForSemanticSegmentation.from_pretrained(
    config.pretrained_model_name,
    num_labels=config.num_classes,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
model.to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"SegFormer-B2 model loaded successfully! Trainable parameters: {total_params / 1e6:.2f}M")

In [ ]:
# Section 6: Evaluation Metrics Calculation (mIoU, Precision, Recall, F1)
from sklearn.metrics import confusion_matrix

def calculate_segmentation_metrics(pred_masks, true_masks, num_classes=5):
    """
    Computes per-class IoU, Mean IoU, Pixel Accuracy, and F1 Score for oil spills.
    """
    preds_flat = pred_masks.flatten()
    trues_flat = true_masks.flatten()
    
    cm = confusion_matrix(trues_flat, preds_flat, labels=list(range(num_classes)))
    ious = []
    for i in range(num_classes):
        intersection = cm[i, i]
        union = cm[i, :].sum() + cm[:, i].sum() - intersection
        iou = intersection / union if union > 0 else 0.0
        ious.append(iou)
        
    pixel_acc = np.diag(cm).sum() / cm.sum() if cm.sum() > 0 else 0.0
    mean_iou = np.mean(ious)
    oil_spill_iou = ious[1] if num_classes > 1 else 0.0
    
    return {
        "pixel_accuracy": float(pixel_acc),
        "mean_iou": float(mean_iou),
        "oil_spill_iou": float(oil_spill_iou),
        "per_class_iou": [float(x) for x in ious],
        "confusion_matrix": cm
    }

print("Evaluation metrics module compiled.")

In [ ]:
# Section 7: Complete Training Loop with AMP, Checkpointing, & Early Stopping
import time
from torch.cuda.amp import autocast, GradScaler

def train_epoch(model, dataloader, optimizer, criterion, scaler, device):
    model.train()
    running_loss = 0.0
    for batch in dataloader:
        images = batch["pixel_values"].to(device)
        masks = batch["labels"].to(device)
        
        optimizer.zero_grad()
        with autocast(enabled=config.use_amp):
            outputs = model(pixel_values=images)
            loss = criterion(outputs.logits, masks)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
    return running_loss / len(dataloader.dataset)

def evaluate(model, dataloader, device, num_classes=5):
    model.eval()
    all_preds = []
    all_trues = []
    with torch.no_grad():
        for batch in dataloader:
            images = batch["pixel_values"].to(device)
            masks = batch["labels"].to(device)
            outputs = model(pixel_values=images)
            logits = F.interpolate(outputs.logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.append(preds)
            all_trues.append(masks.cpu().numpy())
            
    all_preds = np.concatenate(all_preds, axis=0)
    all_trues = np.concatenate(all_trues, axis=0)
    return calculate_segmentation_metrics(all_preds, all_trues, num_classes=num_classes)

print("Training & Validation loop modules compiled.")

In [ ]:
# Section 8: Model Checkpoint Saving & Deployment Packaging
def save_final_checkpoint(model, config, best_epoch, best_miou, best_oil_iou):
    checkpoint_path = os.path.join(config.output_dir, "best_oilspill_segformer.pth")
    save_dict = {
        "epoch": best_epoch,
        "model_state_dict": model.state_dict(),
        "best_val_miou": best_miou,
        "best_oil_spill_iou": best_oil_iou,
        "config": vars(config),
        "class_names": config.class_names,
        "preprocessing": {
            "image_size": config.image_size,
            "mean": [0.485, 0.456, 0.406],
            "std": [0.229, 0.224, 0.225]
        }
    }
    torch.save(save_dict, checkpoint_path)
    print(f"Best model checkpoint saved to: {checkpoint_path}")
    
    # Save HuggingFace formatted artifacts
    hf_dir = os.path.join(config.output_dir, "segformer_hf")
    model.save_pretrained(hf_dir)
    print(f"HuggingFace weights exported to: {hf_dir}")

print("Model export routine ready.")

## Section 9: Execution Instructions for Kaggle / Colab

### Kaggle Instructions:
1. Upload the **M4D Oil Spill Detection Dataset** via `Add Data` -> `Upload Dataset`.
2. Set `TrainingConfig.data_dir = '/kaggle/input/<dataset-slug>'`.
3. Select **GPU T4 x2** or **P100** under `Session Options` -> `Accelerator`.
4. Run all cells.
5. Download the output checkpoint from `/kaggle/working/artifacts/checkpoints/best_oilspill_segformer.pth`.

### Google Colab Instructions:
1. Mount Google Drive: `from google.colab import drive; drive.mount('/content/drive')`.
2. Place dataset in Drive and set `data_dir` accordingly.
3. Select `Runtime` -> `Change runtime type` -> `T4 GPU`.
4. Run all cells.